# Full dataset

In [1]:
from datasets import load_dataset
from collections import Counter

CACHE_DIR = r"D:\Hand_Gesture\data\raw\hf_cache_full"

dataset = load_dataset(
    "testdummyvt/hagRIDv2_512px",
    split="train",
    cache_dir=CACHE_DIR,
)

label_names = dataset.features["label"].names
counts = Counter(dataset["label"])

print("Label names:")
for i, name in enumerate(label_names):
    print(i, name)

print("\nClass counts:")
for idx, count in sorted(counts.items()):
    print(idx, label_names[idx], count)

README.md: 0.00B [00:00, ?B/s]

c:\Users\Eric Yu\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Eric Yu\.cache\huggingface\hub\datasets--testdummyvt--hagRIDv2_512px. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Resolving data files:   0%|          | 0/1503 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/184 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/301 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1503 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/184 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/301 [00:00<?, ?it/s]

PermissionError: [WinError 21] 裝置未就緒。: 'D:\\'

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
from tqdm import tqdm
import random

label_names = dataset.features["label"].names
labels = dataset["label"]

target_map = {
    "fist": 1,
    "like": 2,
    "ok": 3,
    "one": 4,
    "palm": 5,
}

TARGET_PER_CLASS = 1000
NA_COUNT = 5000

out_dir = Path("D:/Hand_Gesture/data/processed_full_10k")
crop_dir = out_dir / "crops"
lm_dir = out_dir / "landmarks"

crop_dir.mkdir(parents=True, exist_ok=True)
lm_dir.mkdir(parents=True, exist_ok=True)

selected_indices = []

for name, new_label in target_map.items():
    original_label_id = label_names.index(name)
    idxs = [i for i, y in enumerate(labels) if y == original_label_id]
    selected_indices += [(idx, new_label, name) for idx in idxs[:TARGET_PER_CLASS]]

target_label_ids = {label_names.index(name) for name in target_map.keys()}
na_indices = [i for i, y in enumerate(labels) if y not in target_label_ids]

random.seed(0)
na_indices = random.sample(na_indices, NA_COUNT)

selected_indices += [(idx, 0, "N_A") for idx in na_indices]

print("total selected:", len(selected_indices))

total selected: 10000


In [ ]:
from hand_preprocess import MediaPipeHandPreprocessor

preprocessor = MediaPipeHandPreprocessor()

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
from tqdm import tqdm

out_dir = Path("D:/Hand_Gesture/data/processed_full_10k")
crop_dir = out_dir / "crops"
lm_dir = out_dir / "landmarks"

crop_dir.mkdir(parents=True, exist_ok=True)
lm_dir.mkdir(parents=True, exist_ok=True)

records = []
fail_count = 0

for idx, new_label, new_label_name in tqdm(selected_indices):
    sample = dataset[idx]
    img = sample["image"]
    original_label = sample["label"]
    original_label_name = label_names[original_label]

    result = preprocessor.preprocess_image(img)

    if result is None:
        fail_count += 1
        continue

    crop, landmarks = result

    base_name = f"{idx}_{new_label_name}"

    crop_path = crop_dir / f"{base_name}.png"
    lm_path = lm_dir / f"{base_name}.npy"

    Image.fromarray(crop).save(crop_path)
    np.save(lm_path, landmarks)

    records.append({
        "idx": idx,
        "original_label": original_label,
        "original_label_name": original_label_name,
        "label": new_label,
        "label_name": new_label_name,
        "crop_path": str(crop_path),
        "landmark_path": str(lm_path),
        "quality": "ok"
    })

df = pd.DataFrame(records)
df.to_csv(out_dir / "labels_fixed.csv", index=False)

print("selected:", len(selected_indices))
print("saved:", len(df))
print("failed no hand:", fail_count)
print("labels_fixed.csv:", out_dir / "labels_fixed.csv")

100%|██████████| 10000/10000 [03:52<00:00, 42.95it/s]

selected: 10000
saved: 9207
failed no hand: 793
labels.csv: D:\Hand_Gesture\data\processed_full_10k\labels.csv
